# v8: in-place sharp turns, based on v4 repaired

This version deliberately starts from v4_repaired, the working baseline. Run one cell at a time on the Jetson. Driving stays commented out. Stop previous camera kernels before opening the camera here, and copy your reliable motor/vision calibration.

Changes:
- Ordinary sharp-turn and lost-line search commands now reverse the inner wheel: left turn = (-TURN_SPEED, +TURN_SPEED), right turn = (+TURN_SPEED, -TURN_SPEED).
- Green turns already reversed the inner wheel in v4. Their timed forward advance is now zero, so after the brief stop they pivot from the current position.
- Original three-band detection, ordinary following, gap crossing, marker confirmation, BOTH-marker U-turn, cooldown and timeout behavior are retained.

Equal opposite wheel commands aim to rotate about the axle midpoint with little forward travel. They do not locate the corner under the axle or measure 90 degrees. The camera detects a large line error/green instruction, not a verified 90-degree angle. Turn completion still uses v4 line reacquisition and its existing minimum-time guard for green turns.

For an on-line pivot, calibrate GREEN_TRIGGER_Y against where the axle actually is when the marker confirms. Larger values trigger closer/later; smaller values earlier. Ordinary TURN_ENTER is an error threshold, not a distance-to-corner setting. Camera mounting, traction and motor asymmetry affect where the pivot occurs. If removing the advance makes green turns start too early, first inspect the axle position against the corner before adjusting the trigger. Keep a physical stop accessible.

## 1. Initialize camera once


In [ ]:
import time
import cv2 #카메라
import numpy as np # 
import traitlets #값 바뀔 때 신호 연동
import ipywidgets as widgets #화면 위젯

from IPython.display import display #
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) #camera가 보여주는 영상 만들 공간

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)


In [ ]:
latest = {"sample": None} #가장 최근에 카메라가 인식했던 영상 사진 

def receive_frame(change): #
    latest["sample"] = (change["new"], time.monotonic()) #새 장면 찍힐 때 마다 이미지와 시간

camera.observe(receive_frame, names="value") #카메라가 새로운 장면 보일 때 마다 리시브 프레임 실행해서 저장

time.sleep(0.5) #카메라 콜백 시간

if latest["sample"] is None:
    print("No camera frame yet.")
else:
    print("Camera frame received.")


## 2. Black line and green detection
Green mask should show only physical tape markers. Drawn overlay dots are never fed into detection. Tune HSV, area bounds, and trigger position while stationary.


In [ ]:
BLACK_THRESHOLD = 70  # Copy your calibrated threshold here.
# Fractions of image height: change these if the far strip is above the floor.
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]

def detect_curve(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    overlay = frame.copy()
    positions = {}
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        positions[name] = None
        if contours:
            contour = max(contours, key=cv2.contourArea)
            fraction = cv2.contourArea(contour) / float(strip.size)
            moments = cv2.moments(contour)
            # Permit a broad horizontal segment, but reject an almost dark strip.
            if 0.005 < fraction < 0.80 and moments["m00"] > 0:
                cx = moments["m10"] / moments["m00"]
                cy = int(moments["m01"] / moments["m00"]) + y0
                positions[name] = (cx - width / 2.0) / (width / 2.0)
                cv2.circle(overlay, (int(cx), cy), 4, (0, 255, 0), -1)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
        cv2.putText(overlay, name, (3, y0 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)
    cv2.line(overlay, (width // 2, 0), (width // 2, height - 1), (0, 255, 255), 1)
    return positions, overlay, mask

# OpenCV HSV hue range is 0..179. Tune these using the actual green tape.
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.65  # Require a nearby marker before initiating a turn.
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0  # Pixels, at the configured 224px resolution.
GREEN_PAIR_Y_TOLERANCE = 0.15

def detect_green(frame):
    height, width = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    mask[:int(height * GREEN_ROI_TOP), :] = 0
    mask[int(height * GREEN_ROI_BOTTOM):, :] = 0
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0:
            continue
        if not 0.4 <= w / float(h) <= 2.5:
            continue
        moments = cv2.moments(contour)
        if moments['m00'] <= 0:
            continue
        markers.append(dict(x=moments['m10']/moments['m00'],
                            y=moments['m01']/moments['m00'],
                            area=area, w=w, h=h, box=(x,y,w,h)))
    markers.sort(key=lambda item: item['area'], reverse=True)
    return markers, mask

def get_line_points(positions, width, height):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in ('far', 'middle', 'near') if positions.get(name) is not None]

def classify_green_direction(positions, markers, width, height):
    if not markers:
        return 'NONE'
    points = get_line_points(positions, width, height)
    if len(points) < 2:
        return 'UNKNOWN'
    near_x, near_y = points[-1]
    dx, dy = points[0][0] - near_x, points[0][1] - near_y
    length = (dx * dx + dy * dy) ** 0.5
    if length < 1e-6:
        return 'UNKNOWN'
    dx, dy = dx / length, dy / length
    sides = {'LEFT': [], 'RIGHT': []}
    for marker in markers:
        if marker['y'] < height * GREEN_TRIGGER_Y:
            continue
        score = (marker['x'] - near_x) * dy - (marker['y'] - near_y) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides['LEFT' if score > 0 else 'RIGHT'].append(marker)
    if sides['LEFT'] and sides['RIGHT']:
        paired = any(abs(a['y'] - b['y']) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides['LEFT'] for b in sides['RIGHT'])
        return 'BOTH' if paired else 'UNKNOWN'
    if sides['LEFT']:
        return 'LEFT'
    if sides['RIGHT']:
        return 'RIGHT'
    return 'CENTER'


## 3. Existing curve controller
Use measured reliable motor commands. Default curve speed is lower than base speed; raise it if a wheel stalls.


In [ ]:
# Copy your measured, reliable motor settings. These are initial guesses.
BASE_SPEED = 0.15
CURVE_SPEED = 0.13  # Must still reliably move the robot on the floor.
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15
ALLOW_REVERSE = True  # V8: reverse the inner wheel during ordinary sharp turns and searches.
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 1.0
LOST_TIMEOUT = 0.35
FRAME_TIMEOUT = 0.5

def clamp(value, low, high):
    return max(low, min(high, value))

class CurveController:
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed if ALLOW_REVERSE else 0.0
        return (speed, inner) if direction > 0 else (inner, speed)

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if self.turn_started is None:
                self.turn_started = now
                self.turn_direction = self.last_direction
            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}
        error = sum(weights[name] * value for name, value in visible) / sum(weights[name] for name, _ in visible)
        farthest = positions.get("far")
        if farthest is None:
            farthest = positions.get("middle")
        # Anticipate a bend from its displacement relative to the near strip.
        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)
        error = clamp(error, -1.0, 1.0)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            # Require the near line and combined direction to agree before exiting.
            if near is not None and abs(near) < TURN_EXIT and abs(error) < TURN_EXIT:
                self.turn_started = None
                self.turn_direction = 0
            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, "FOLLOW"


## 4. Gap controller
Copy your calibrated gap timeout and speed here.


In [ ]:
# Initial trial values; calibrate on the real floor at your working motor power.
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.0       # Maximum time driving without confirmed near/middle tape.
GAP_STEER_LIMIT = 0.015 # Only a small remembered correction; not heading feedback.
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25   # Normalized offset from image centre.
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCurveController(CurveController):
    def __init__(self):
        super().__init__()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        middle = positions.get("middle")
        values = [value for value in positions.values() if value is not None]

        if self.gap_started is not None:
            # Never reset this deadline because of an isolated noisy detection.
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.stop("gap crossing timed out")
            if near is not None and middle is not None:
                self.confirm_frames += 1
            else:
                self.confirm_frames = 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.lost_started = None
            self.last_direction = 0
            # Resume the existing controller using the newly observed tape.

        if not values:
            recent_straight = (self.straight_seen is not None and
                               now - self.straight_seen <= GAP_APPROACH_MAX_AGE)
            if (self.turn_started is None and recent_straight and
                    self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return super().step(positions, now)

        left, right, state = super().step(positions, now)
        aligned = max(abs(value) for value in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        # A short single-band transition can occur as the line exits the image.
        # Preserve earlier evidence only until GAP_APPROACH_MAX_AGE expires.
        return left, right, state


## 5. Green turn state machine
Three matching processed frames request STOP → TURN → FOLLOW with the default zero advance. The existing optional ADVANCE state remains available through GREEN_ADVANCE_TIME, but commands no forward movement at zero. Both left/right turns and the original BOTH-marker U-turn reverse the inner wheel. Minimum turn duration is a reacquisition guard, not a measured angle. Timeout stops and marker cooldown remain unchanged from v4.


In [ ]:
# Initial trial values, not calibrated turn angles or distances.
GREEN_CONFIRM_FRAMES = 3
GREEN_STOP_TIME = 0.12
GREEN_ADVANCE_TIME = 0.0  # V8: pivot from the stop position; no timed forward advance.
GREEN_ADVANCE_SPEED = BASE_SPEED
GREEN_TURN_SPEED = 0.15
GREEN_TURN_MIN_TIME = 0.25
GREEN_TURN_TIMEOUT = 1.5
UTURN_MIN_TIME = 0.8
UTURN_TIMEOUT = 2.5
GREEN_REACQUIRE_FRAMES = 3
GREEN_CLEAR_FRAMES = 5
GREEN_COOLDOWN = 0.8

def line_centered(positions):
    return all(positions.get(name) is not None and abs(positions[name]) < TURN_EXIT
               for name in ('near', 'middle'))

class GreenNavigator:
    def __init__(self):
        self.line = GapCurveController()
        self.state = 'FOLLOW'
        self.state_started = None
        self.direction = None
        self.candidate = None
        self.candidate_frames = 0
        self.armed = True
        self.clear_frames = 0
        self.cooldown_until = 0.0
        self.departed = False
        self.reacquire_frames = 0
        self.stopped = False
        self.reason = ''

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        return 0.0, 0.0, 'STOP: ' + reason

    def enter(self, state, now):
        self.state = state
        self.state_started = now

    def turn(self):
        speed = clamp(GREEN_TURN_SPEED, 0.0, MAX_SPEED)
        if self.direction == 'RIGHT':
            return speed, -speed, 'GREEN_TURN_RIGHT'
        return -speed, speed, 'GREEN_U_TURN' if self.direction == 'BOTH' else 'GREEN_TURN_LEFT'

    def step(self, positions, direction, now):
        if self.stopped:
            return self.stop(self.reason)

        if self.state == 'GREEN_STOP':
            if now - self.state_started < GREEN_STOP_TIME:
                return 0.0, 0.0, 'GREEN_STOP'
            self.enter('GREEN_ADVANCE', now)

        if self.state == 'GREEN_ADVANCE':
            if now - self.state_started < GREEN_ADVANCE_TIME:
                speed = clamp(GREEN_ADVANCE_SPEED, 0.0, MAX_SPEED)
                return speed, speed, 'GREEN_ADVANCE'
            self.departed = False
            self.reacquire_frames = 0
            self.enter('GREEN_TURN', now)

        if self.state == 'GREEN_TURN':
            elapsed = now - self.state_started
            limit = UTURN_TIMEOUT if self.direction == 'BOTH' else GREEN_TURN_TIMEOUT
            minimum = UTURN_MIN_TIME if self.direction == 'BOTH' else GREEN_TURN_MIN_TIME
            if elapsed >= limit:
                return self.stop('green turn timed out')
            centered = line_centered(positions)
            if not centered:
                self.departed = True
            if self.departed and elapsed >= minimum and centered:
                self.reacquire_frames += 1
            else:
                self.reacquire_frames = 0
            if self.reacquire_frames < GREEN_REACQUIRE_FRAMES:
                return self.turn()
            # Remove stale line-turn and gap history before handing control back.
            self.line = GapCurveController()
            self.enter('FOLLOW', now)
            self.cooldown_until = now + GREEN_COOLDOWN
            self.clear_frames = 0
            return 0.0, 0.0, 'GREEN_DONE'

        if not self.armed:
            # Require actual absence, not UNKNOWN/CENTER, to clear the old marker.
            self.clear_frames = self.clear_frames + 1 if direction == 'NONE' else 0
            if now >= self.cooldown_until and self.clear_frames >= GREEN_CLEAR_FRAMES:
                self.armed = True

        # Do not initiate another manoeuvre in the middle of a gap or sharp turn.
        eligible = (self.armed and self.line.gap_started is None and
                    self.line.turn_started is None and direction in ('LEFT','RIGHT','BOTH'))
        if eligible:
            if direction == self.candidate:
                self.candidate_frames += 1
            else:
                self.candidate, self.candidate_frames = direction, 1
            if self.candidate_frames >= GREEN_CONFIRM_FRAMES:
                self.direction = direction
                self.armed = False
                self.clear_frames = 0
                self.candidate = None
                self.candidate_frames = 0
                self.enter('GREEN_STOP', now)
                return 0.0, 0.0, 'GREEN_STOP'
        else:
            self.candidate = None
            self.candidate_frames = 0

        left, right, state = self.line.step(positions, now)
        if self.line.stopped:
            return self.stop(state)
        return left, right, state


## 6. Runner
Preview computes commands but only stops motors. Every run creates fresh controller state. Camera frames are checked for freshness even when no new frame arrives.


In [ ]:
overlay_view = widgets.Image(format='jpeg', width=336, height=336)
mask_view = widgets.Image(format='jpeg', width=336, height=336)
status_view = widgets.Label(value='Ready for stationary preview')
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_v8(seconds=10, drive=False):
    robot.stop()
    controller = GreenNavigator()
    deadline = time.monotonic() + seconds
    previous_stamp = None
    next_display = 0.0
    try:
        while time.monotonic() < deadline:
            sample = latest['sample']
            if sample is None:
                print('Stopped: no camera frame')
                break
            frame, captured_at = sample
            now = time.monotonic()
            # Check freshness BEFORE skipping repeated frames.
            if now - captured_at > FRAME_TIMEOUT:
                print('Stopped: stale camera frame')
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            positions, overlay, _ = detect_curve(frame)
            markers, green_mask = detect_green(frame)
            height, width = frame.shape[:2]
            direction = classify_green_direction(positions, markers, width, height)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print('Stopped: deadline or slow image processing')
                break
            left, right, state = controller.step(positions, direction, now)
            if drive:
                robot.set_motors(left, right)
            if now >= next_display:
                points = get_line_points(positions, width, height)
                for a,b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int,a)), tuple(map(int,b)), (255,0,255), 2)
                for marker in markers:
                    x,y,w,h = marker['box']
                    cv2.rectangle(overlay, (x,y), (x+w,y+h), (0,255,0), 2)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = '%s | %s | marker=%s | L=%.2f R=%.2f' % (
                    'DRIVE' if drive else 'PREVIEW',state,direction,left,right)
                next_display = now + 0.1
            if controller.stopped:
                print(state)
                break
            time.sleep(0.01)
    finally:
        robot.stop()
        print('Run ended; motors stopped.')

def run_preview(seconds=20):
    run_v8(seconds=seconds, drive=False)

def run_robot(seconds=2):
    run_v8(seconds=seconds, drive=True)


## 7. Stationary preview
Test no marker, left, right, and both. Move/rotate the robot by hand to simulate the manoeuvre. A timeout stops preview too; reposition and run again.


In [ ]:
run_preview(seconds=20)


## 8. Short driving test
First test one unmarked sharp left corner, then right, then green-marked left/right corners. Watch whether the axle midpoint is over the tape/corner when rotation starts. Preview shows negative inner-wheel commands during turns. The original BOTH-marker U-turn remains supported. Retest straight lines and gaps after copying motor calibration.

Start with a short run and extend the overall run duration only as needed to observe a complete maneuver. Keep the failure timeouts bounded. If it rotates in the correct place but overshoots, check TURN_SPEED / GREEN_TURN_SPEED and line reacquisition; if it starts in the wrong place, inspect the trigger/camera geometry. No physical 90-degree or exact pivot position is guaranteed by this camera-only controller.


In [ ]:
# Explicitly uncomment to move the robot:
# run_robot(seconds=2)


## 9. Cleanup
Run before switching notebooks. Shut down this kernel afterward.


In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()
